# 04 · RAG, answering from your own documents

**AI Fundamentals in 3 Hours** · Data Sense

The model has never seen your company's documents. RAG fixes that, not by teaching the
model anything, but by **finding the right paragraph and pasting it into the prompt**.

In this notebook:

1. Watch the model confidently invent facts about your company
2. Load → split → embed → store, with LangChain's four components
3. Search by meaning instead of keywords
4. Build the augmented prompt **and look at it**
5. Get grounded answers with citations, and a model that says "I don't know"
6. See where retrieval fails, and what to reach for


In [ ]:
# --- run this first, in every notebook ---
import os, json
from pathlib import Path

# read keys out of .env (works from the repo root or from notebooks/)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        for line in candidate.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1-mini"          # provider:model - change this one string to switch providers
model = init_chat_model(MODEL, temperature=0)

import textwrap
def wrap(text, width=88):
    """Print long text wrapped, so answers stay readable on a projector."""
    print(textwrap.fill(str(text), width=width))

assert os.environ.get("OPENAI_API_KEY"), "No API key found - check your .env file"
print("ready |", MODEL)

## 1. First, the problem

Our three documents in `../data/` are internal policies for a made-up company, Nimbus Retail.
The model has never seen them. Let's ask anyway.

In [ ]:
QUESTION = "How long does Nimbus Retail take to process a refund after they receive my return?"

wrap(model.invoke(QUESTION).content)

Read that carefully. Fluent, confident, well-structured, and **completely invented**.
It is describing a generic e-commerce policy, not ours.

The real answer is in `data/refund_policy.md`. The model simply cannot see it.

> No amount of prompt engineering fixes this. The fix is to **put the document in the
> prompt** which is all RAG is.

## 2. Load

A `Document` is LangChain's unit of content: `page_content` plus `metadata`. Every loader
in the ecosystem. PDF, Notion, S3, Confluence, a website, produces these same objects,
so everything downstream stays identical.

In [ ]:
from langchain_core.documents import Document

DATA = Path("../data") if Path("../data").exists() else Path("data")

docs = [
    Document(page_content=p.read_text(), metadata={"source": p.name})
    for p in sorted(DATA.glob("*.md"))
]

for d in docs:
    print(f"{d.metadata['source']:<22}{len(d.page_content):>6} chars  ~{len(d.page_content)//4:>5} tokens")

print()
print("Small enough to paste whole today. Not at 10,000 documents - and pasting")
print("irrelevant text makes answers worse, not just more expensive.")

## 3. Split

Two reasons to chunk: a whole document may not fit, and retrieving a whole document buries
the one line that mattered.

`RecursiveCharacterTextSplitter` tries each separator in order, paragraphs before lines,
lines before words, so it breaks at the most natural boundary it can find. The `chunk_overlap`
repeats a little text between neighbours, so a sentence split across a boundary survives.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n## ", "\n\n", "\n", " ", ""],   # markdown headings first
    add_start_index=True,                           # keeps where each chunk came from
)

chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents -> {len(chunks)} chunks\n")
for c in chunks[:5]:
    head = c.page_content.strip().splitlines()[0][:44]
    print(f"[{c.metadata['source']:<20} @{c.metadata['start_index']:>5}] {len(c.page_content):>4} chars  {head}")
print("...")

### Always look at your chunks

Bad retrieval is usually bad chunking, and you can see it by eye in ten seconds.

In [ ]:
print(chunks[2].page_content)

## 4. Embed and store

An embedding turns text into a list of floats positioned so that **similar meanings land
close together**. We embed every chunk once, up front, this is the "indexing" lane from the
slide, and it runs offline, not on every request.

Embedding is cheap: about **\$0.02 per million tokens**, roughly 1/20th of the cheapest
chat model.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# one line: embeds every chunk and stores the vectors alongside the original text
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

sample = embeddings.embed_query("how long does a refund take")
print("a query embedding is", len(sample), "floats")
print("first eight:", [round(x, 3) for x in sample[:8]])
print()
print("Nobody reads these numbers. We only ever ask: which is closest to which?")

> **The most common silent RAG bug:** indexing with one embedding model and querying with
> another. No error, just garbage results. Here the vector store holds the embeddings object,
> so it is impossible to get wrong, one more thing the framework takes off your plate.

## 5. Search

In [ ]:
for score, doc in [(s, d) for d, s in vectorstore.similarity_search_with_score(QUESTION, k=4)]:
    head = doc.page_content.strip().splitlines()[0][:48]
    print(f"{score:.3f}  {doc.metadata['source']:<22}{head}")

### Why this beats keyword search

Try questions that share **no words** with the documents.

In [ ]:
for q in ["can I get my money back", "when will my parcel show up", "my payment bounced"]:
    print(f"{q!r}")
    for doc, score in vectorstore.similarity_search_with_score(q, k=2):
        head = doc.page_content.strip().splitlines()[0][:46]
        print(f"   {score:.3f}  {doc.metadata['source']:<20}{head}")
    print()

print("Not one of those queries uses the words the documents use.")
print("No keyword search would have found any of them.")

Look closely at the first one. `"can I get my money back"` matches the *payment reversal*
section about as strongly as the refund policy, semantically close, practically the wrong
document.

That near-miss is not a bug in this notebook. It is what semantic search does, and it is
exactly what hybrid search and reranking exist to fix.

## 6. The retriever

`as_retriever()` wraps the store in the standard **Runnable** interface, the same `.invoke`
you have used on models and structured output. That uniformity is what lets you drop a
retriever into a chain or hand it to an agent later without changing anything around it.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

found = retriever.invoke("what is the free shipping threshold")
print(f"{len(found)} documents returned\n")
print(found[0].page_content[:220])

## 7. Build the augmented prompt, and look at it

This is the most important cell in the notebook. Every RAG framework in existence,
however many layers deep, ends up producing a string like this one.

In [ ]:
SYSTEM = """You answer questions about Nimbus Retail using ONLY the context provided.
If the answer is not in the context, say "I don't know - that isn't in our documentation."
Never guess. Cite the source of every claim using the [source] label shown in the context."""


def format_context(found: list[Document]) -> str:
    return "\n\n".join(
        f"[{d.metadata['source']}]\n{d.page_content}" for d in found
    )


found = retriever.invoke(QUESTION)
user_message = f"<context>\n{format_context(found)}\n</context>\n\nQuestion: {QUESTION}"

print(user_message[:1500])
print("\n... [truncated for display]")

**That is RAG.** Retrieved text, pasted into the prompt, with an instruction to stay inside
it. No fine-tuning, no model change, no magic. You could have typed that string by hand.

## 8. Now ask again

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def ask(question: str, k: int = 4, show_sources: bool = True) -> str:
    found = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(question)
    answer = model.invoke([
        SystemMessage(SYSTEM),
        HumanMessage(f"<context>\n{format_context(found)}\n</context>\n\nQuestion: {question}"),
    ]).content

    wrap(answer)
    if show_sources:
        print("\n" + "-" * 58)
        print("retrieved:", ", ".join(sorted({d.metadata["source"] for d in found})))
    return answer


ask(QUESTION);

Compare that with the confident fiction from the very first cell. Same model, same
question. The only difference is **what we put in the prompt**.

### A few more

In [ ]:
for q in [
    "Do you deliver to international addresses?",
    "I paid by card but the order failed and money was debited. What happens?",
    "What is the free shipping threshold?",
]:
    print("=" * 72)
    print("Q:", q, "\n")
    ask(q, show_sources=False)
    print()

## 9. The same thing as a chain

Now that you have seen every step by hand, here is the composed version. The `|` operator
pipes one runnable into the next, retriever, then prompt, then model, then string.

Read it as a sentence: *take the question, retrieve context, fill the prompt, call the model,
return the text.*

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("user", "<context>\n{context}\n</context>\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_context, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

wrap(rag_chain.invoke("How many days do I have to return an item?"))

Five lines, and it has `.invoke`, `.batch` and `.stream` for free, because it is a
runnable like everything else.

In [ ]:
print("streaming, token by token:\n")
for piece in rag_chain.stream("What payment methods do you accept?"):
    print(piece, end="", flush=True)
print()

## 10. The most valuable behaviour: knowing when to shut up

A RAG system that confidently answers questions it has no source for is **worse than no
system at all**, because people will trust it.

One line in our system prompt fixes this. Let's test it.

In [ ]:
print("Q: What is Nimbus Retail's parental leave policy?\n")
ask("What is Nimbus Retail's parental leave policy?")

It refuses. That single instruction is the difference between a demo and something you can
put in front of a paying customer.

## 11. Where retrieval fails

Your first RAG build always works. Here is what breaks next.

In [ ]:
def peek(query, k=3):
    for d, s in vectorstore.similarity_search_with_score(query, k=k):
        head = d.page_content.strip().splitlines()[0][:46]
        print(f"   {s:.3f}  {d.metadata['source']:<20}{head}")

print("FAILURE 1 - exact identifiers")
print("Embeddings carry meaning, and '91204' has none.\n")
peek("order 91204")

print("\n\nFAILURE 2 - the top-k all say the same thing")
print("Ask something broad and near-identical chunks crowd out other useful ones.\n")
peek("refund", k=4)

print("\n\nFAILURE 3 - a confident distractor")
print("Note what account_help.md says about the wifi password.\n")
peek("what is the wifi password")

### The diagnostic that will save you weeks

When a RAG answer is wrong, two completely different bugs look identical from the outside:

| symptom | check | it is a... |
|---|---|---|
| the correct chunk was **not** retrieved | print the hits | **search** problem: hybrid search, reranking, better chunking, metadata filters |
| the correct chunk **was** retrieved, answer still wrong | print the prompt | **generation** problem: prompt wording, model choice, too much context |

Always check retrieval first. Prompt engineering cannot fix a chunk that was never fetched.

The fixes, in the order worth trying:

1. **Hybrid search:** keyword (BM25) plus vector, merged. Biggest single win, almost always.
2. **Reranking:** retrieve 30 candidates, let a cheap cross-encoder pick the best 5.
3. **Query rewriting:** have the LLM rephrase a vague question before searching.
4. **Better chunking:** split on structure, keep headings with their body.
5. **Metadata filters:** narrow by product, date or language before searching at all.

## 12. Going to production is one line

We used `InMemoryVectorStore` so you could see everything. Everything downstream, the
retriever, the chain, the agent in notebook 06, talks to the **same interface**, so
swapping in a real database changes exactly one line:

```python
# development
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

# production - pip install langchain-chroma
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

# managed - pip install langchain-pinecone
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore.from_documents(chunks, embeddings, index_name="nimbus")
```

That is the argument for a framework in one code block.

## 13. Your turn

1. **Break it.** Set `chunk_size=200`, rebuild the store, rerun the refund question. Watch
   answers get worse as chunks lose their surrounding context.

2. **Your own documents.** Drop your own notes or a PDF's text into `data/` and rerun the
   notebook. This is the fastest way to make RAG feel real.

3. **Tune k.** Try `k=1` and `k=10` on the same question. Notice that more context is not
   better, and costs more.

4. **Measure it.** Write ten questions with known correct source documents. Score how often
   the right document lands in the top 3. Congratulations, you just built an eval set,
   which is the actual job.


In [ ]:
# your turn - scratch cell
